In [6]:
import torch
import patcher
import numpy as np
import xarray as xr
from datetime import datetime, timedelta

start_date = datetime(2024, 8, 1)
end_date = datetime(2026, 6, 30)
context = patcher.Context('../db', 1)

dates = []
reqs = []
current_date = start_date
while current_date < end_date:
    dates.append(current_date)
    reqs.append(patcher.Request('sd', 720, 0, current_date.strftime('%Y%m%d'), 360, 90, 1, 4, 1))
    current_date += timedelta(days=1)

swe_anom = torch.concatenate(patcher.load(context, reqs), dim=0)
swe_clim = torch.concatenate(patcher.load_climate(context, reqs), dim=0)
swe_anom = (torch.exp(swe_anom + swe_clim) - torch.exp(swe_clim)).numpy().astype(np.float32)
swe_clim = swe_clim.numpy().astype(np.float32)
swe_anom[swe_clim < 0.01] = torch.nan
swe_anom[swe_clim > 500] = torch.nan

lat = np.arange(0, 90)
lon = np.arange(0, 360)
times = np.array(dates, dtype='datetime64[D]')

T = swe_anom.shape[0]
W = 7

kernel = np.ones(W, dtype=np.float32) / W
anom_w = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='valid'), axis=0, arr=swe_anom)
clim_w = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='valid'), axis=0, arr=swe_clim)
times_w = times[:T - W + 1]

months = np.array([d.year * 100 + d.month for d in dates])
unique_months = np.unique(months)

anom_m_list, clim_m_list, times_m_list = [], [], []
for m in unique_months:
    mask = months == m
    anom_m_list.append(swe_anom[mask].mean(axis=0))
    clim_m_list.append(swe_clim[mask].mean(axis=0))
    times_m_list.append(times[mask][0])

anom_m = np.stack(anom_m_list, axis=0)
clim_m = np.stack(clim_m_list, axis=0)
times_m = np.array(times_m_list, dtype='datetime64[D]')

ds = xr.Dataset(
    data_vars={
        'swe_anom_daily': (('time_daily', 'lat', 'lon'), swe_anom),
        'swe_clim_daily': (('time_daily', 'lat', 'lon'), swe_clim),
        'swe_anom_weekly': (('time_weekly', 'lat', 'lon'), anom_w),
        'swe_clim_weekly': (('time_weekly', 'lat', 'lon'), clim_w),
        'swe_anom_monthly': (('time_monthly', 'lat', 'lon'), anom_m),
        'swe_clim_monthly': (('time_monthly', 'lat', 'lon'), clim_m),
    },
    coords={
        'time_daily': times,
        'time_weekly': times_w,
        'time_monthly': times_m,
        'lat': lat,
        'lon': lon,
    }
)

ds.to_netcdf('../data/score/swe/era.nc', engine='h5netcdf',
             encoding={v: {'zlib': True, 'complevel': 4, 'shuffle': True} for v in ds.data_vars})

In [1]:
from pathlib import Path
import torch
import patcher
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

directory = Path('../data/score/swe/inm')
directory.mkdir(parents=True, exist_ok=True)

start_date = datetime(2024, 9, 1)
end_date = datetime(2026, 4, 30)
context = patcher.Context('../db', 1)

lat = np.arange(0, 90)
lon = np.arange(-180, 180)
W = 7

current_date = start_date
while current_date < end_date:
    req = patcher.Request('inm_swe', 0, 0, current_date.strftime('%Y%m%d'), 360, 90, 124, 1, 1, current_date.month)
    swe_anom = patcher.load(context, [req])[0].mean(dim=0)
    idx = torch.isnan(swe_anom).any(dim=(1, 2)).nonzero(as_tuple=True)[0]
    T = idx[0].item() if idx.numel() > 0 else None
    swe_anom = swe_anom[:T]

    clim_req = patcher.Request('sd', 0, 0, current_date.strftime('%Y%m%d'), 360, 90, T, 4, 1)
    swe_clim = patcher.load_climate(context, [clim_req])[0]
    swe_anom = torch.exp(swe_anom + swe_clim) - 1
    swe_anom = swe_anom.clamp(0, 5000) - torch.exp(swe_clim) + 1
    mask = (torch.exp(swe_clim) > 500) | (torch.exp(swe_clim) < 0.1)
    swe_anom[mask] = torch.nan
    swe_anom = swe_anom.numpy().astype(np.float32)

    dates = [current_date + timedelta(days=i) for i in range(T)]
    times = np.array(dates, dtype='datetime64[D]')

    kernel = np.ones(W, dtype=np.float32) / W
    swe_w = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='valid'), axis=0, arr=swe_anom)
    times_w = times[:T - W + 1]

    months = np.array([d.year * 100 + d.month for d in dates])
    unique_months = np.unique(months)

    swe_m_list, times_m_list = [], []
    for m in unique_months:
        mask_m = months == m
        swe_m_list.append(swe_anom[mask_m].mean(axis=0))
        times_m_list.append(times[mask_m][0])

    swe_m = np.stack(swe_m_list, axis=0)
    times_m = np.array(times_m_list, dtype='datetime64[D]')

    ds = xr.Dataset(
        data_vars={
            'swe_anom_daily': (('time_daily', 'lat', 'lon'), swe_anom),
            'swe_anom_weekly': (('time_weekly', 'lat', 'lon'), swe_w),
            'swe_anom_monthly': (('time_monthly', 'lat', 'lon'), swe_m),
        },
        coords={
            'time_daily': times,
            'time_weekly': times_w,
            'time_monthly': times_m,
            'lat': lat,
            'lon': lon,
        }
    )

    run_name = current_date.strftime('%Y%m')
    out = directory / f'{run_name}.nc'
    ds.to_netcdf(out, engine='h5netcdf',
        encoding={v: {'zlib': True, 'complevel': 4, 'shuffle': True} for v in ds.data_vars}
    )
    current_date = current_date + relativedelta(months=1)